# Chains

## 传统 Chain 使用方式

### 基础链(废弃)

LLMChain 是 LangChain 早期的核心组件，用于将 LLM 和 Prompt 组合成一个可调用的链。

> **注意**: LLMChain 已被标记为遗留API，推荐使用 LCEL（`prompt | llm`）替代。

In [10]:
from langchain_classic.chains.llm import LLMChain
from langchain_core.prompts import PromptTemplate
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

# 1. 创建大模型实例
llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME", "gpt-3.5-turbo"),
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    openai_api_base=os.getenv("OPENAI_BASE_URL"),
)

# 2. 创建 PromptTemplate
prompt = PromptTemplate(
    input_variables=["topic"],
    template="请用一句话简单介绍{topic}是什么？",
)

# 3. 创建 LLMChain
chain = LLMChain(llm=llm, prompt=prompt)

# 4. 调用 chain
result = chain.invoke({"topic": "LangChain"})
print("LLMChain 输出:")
print(result["text"])

LLMChain 输出:
LangChain是一个用于构建和集成大型语言模型应用的框架，旨在简化开发流程并提升模型的使用效率。


### 顺序链
- SimpleSequentialChain,表示单个输入输出
- SequentialChian,表示多个输入输出

### 数学链
> LLMMathChain

### 路由链
> RouterChain

### 文档链
> StuffDocumentsChain

## LCEL 现代写法（推荐）

使用 `|` 管道符将 prompt 和 llm 组合成 chain，更简洁直观。

In [2]:
from langchain_core.prompts import ChatPromptTemplate

# LCEL 写法：prompt | llm
prompt_lcel = ChatPromptTemplate.from_template("请用一句话简单介绍{topic}是什么？")
chain_lcel = prompt_lcel | llm

# 调用方式相同
result_lcel = chain_lcel.invoke({"topic": "LangChain"})
print("LCEL 输出:")
print(result_lcel.content)

LCEL 输出:
LangChain 是一个用于构建大语言模型（LLM）应用的开源框架，帮助开发者将大模型与外部数据、工具和工作流串联起来，从而快速搭建智能应用。


## 两种方式对比

| 特性 | LLMChain (传统) | LCEL (推荐) |
|------|----------------|-------------|
| 语法 | `LLMChain(llm=llm, prompt=prompt)` | `prompt \| llm` |
| 可读性 | 较低 | 高 |
| 扩展性 | 需要嵌套 Chain | 管道符组合 |
| 流式支持 | 有限 | 原生支持 |
| 状态 | 已废弃 | 当前推荐 |

## SequentialChain 示例

将多个 Chain 串联执行，前一个的输出作为后一个的输入。

## SimpleSequentialChain 示例

SimpleSequentialChain 是最简单的顺序链，每个步骤只有一个输入和一个输出：
- 输入 → Chain1 → 输出1 → Chain2 → 最终输出

In [5]:
from langchain_classic.chains import SimpleSequentialChain
from langchain_core.prompts import PromptTemplate

# 第一个链：生成标题
prompt_title = PromptTemplate(
    input_variables=["topic"],
    template="请为{topic}生成一个吸引人的标题。",
)
chain_title = LLMChain(llm=llm, prompt=prompt_title)

# 第二个链：根据标题写摘要
prompt_summary = PromptTemplate(
    input_variables=["title"],
    template="请根据标题'{title}'写一段50字以内的摘要。",
)
chain_summary = LLMChain(llm=llm, prompt=prompt_summary)

# 创建 SimpleSequentialChain（单输入单输出）
simple_chain = SimpleSequentialChain(
    chains=[chain_title, chain_summary],
    verbose=True,
)

# 执行：只需要传入最初的输入
result = simple_chain.invoke({"input": "Python编程入门"})
print("最终输出:")
print(result["output"])



> Entering new SimpleSequentialChain chain...
**“零基础也能轻松上手！Python编程入门：从代码到实战，开启你的编程之旅！”**
本文介绍Python编程入门，从零基础到实战，帮助初学者快速掌握编程技能，开启你的编程之旅！

> Finished chain.
最终输出:
本文介绍Python编程入门，从零基础到实战，帮助初学者快速掌握编程技能，开启你的编程之旅！


## SimpleSequentialChain vs SequentialChain

| 特性 | SimpleSequentialChain | SequentialChain |
|------|----------------------|-----------------|
| 输入/输出 | 每步单输入单输出 | 支持多输入多输出 |
| 参数 | `input` | `input_variables` |
| 返回值 | `output` | `output_variables` |
| 适用场景 | 简单串联 | 复杂数据流 |

In [9]:
from langchain_classic.chains import SequentialChain
from langchain_core.prompts import PromptTemplate

# 第一个链：生成大纲
prompt_outline = PromptTemplate(
    input_variables=["topic"],
    template="请为{topic}写一个简短的大纲，包含3个要点。",
)
chain_outline = LLMChain(llm=llm, prompt=prompt_outline, output_key="outline")

# 第二个链：根据大纲写简介
prompt_intro = PromptTemplate(
    input_variables=["outline"],
    template="根据以下大纲写一段简短的介绍：\n{outline}",
)
chain_intro = LLMChain(llm=llm, prompt=prompt_intro, output_key="intro")

# 串联两个链
sequential_chain = SequentialChain(
    chains=[chain_outline, chain_intro],
    input_variables=["topic"],
    output_variables=["outline", "intro"],
    verbose=True,
)

# 执行
result = sequential_chain.invoke({"topic": "Python编程"})
print("大纲:")
print(result["outline"])
print("\n介绍:")
print(result["intro"])



> Entering new SequentialChain chain...

> Finished chain.
大纲:
### Python编程简短大纲

1. **基础语法与数据类型**  
   - 变量与命名规则  
   - 基本数据类型：整数、浮点数、字符串、布尔值  
   - 运算符：算术运算符、比较运算符、逻辑运算符  
   - 输入输出：`input()` 和 `print()` 函数  

2. **控制结构与流程**  
   - 条件语句：`if`、`elif`、`else`  
   - 循环语句：`for` 循环、`while` 循环  
   - 列表推导式：简洁高效的列表生成方式  

3. **函数与模块**  
   - 定义与调用函数  
   - 参数与返回值  
   - 模块的使用：导入标准库和第三方库  
   - 常用内置函数：`len()`、`range()`、`sorted()` 等  

通过以上三个要点，可以快速掌握Python的基础编程知识，并为进一步学习打下坚实基础。

介绍:
Python编程是一门简单易学且功能强大的语言，适合初学者快速上手。本大纲从基础语法与数据类型入手，帮助你理解变量、数据类型以及基本运算符的使用；接着通过控制结构和流程的学习，掌握条件判断和循环操作；最后，通过函数与模块的应用，了解如何封装代码并利用外部库扩展功能。通过这三个核心部分的学习，你将能够灵活运用Python解决实际问题，并为进一步深入学习打下坚实的基础。


## LCEL 等效写法

使用 LCEL 的 `RunnablePassthrough` 和 `RunnableLambda` 实现同样的功能。

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# LCEL 写法：管道符串联
outline_prompt = ChatPromptTemplate.from_template("请为{topic}写一个简短的大纲，包含3个要点。")
intro_prompt = ChatPromptTemplate.from_template("根据以下大纲写一段简短的介绍：\n{outline}")

# 使用 LCEL 组合
chain_lcel = (
    {"topic": RunnablePassthrough()}
    | outline_prompt
    | llm
    | (lambda x: {"outline": x.content})
    | intro_prompt.partial(topic="Python编程")  # 这里简化处理
)

# 更清晰的 LCEL 写法
def extract_outline(response):
    return {"outline": response.content}

chain_lcel_clear = (
    outline_prompt | llm | extract_outline | intro_prompt | llm
)

result_lcel = chain_lcel_clear.invoke({"topic": "Python编程"})
print("LCEL SequentialChain 输出:")
print(result_lcel.content)